# 混合检索与 Rerank：LangChain

流程：BM25 关键词检索 + Chroma 向量检索 → 合并候选文档 → Cross-Encoder Reranker 重排序。

In [ ]:
# 如果环境中没有 rank_bm25，可先执行：
# %pip install rank_bm25 sentence-transformers

import os
from dotenv import load_dotenv
from sentence_transformers import CrossEncoder
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, UnstructuredMarkdownLoader
from langchain_community.retrievers import BM25Retriever
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
knowledge_path = "../knowledge_db/prompt_engineering"
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.environ["GEMINI_API_KEY"],
    temperature=0,
)
embedding_model = HuggingFaceEmbeddings(model_name="moka-ai/m3e-base")
# CrossEncoder 是 Sentence Transformers 提供的模型加载类 然后判断它们的相关性
reranker = CrossEncoder("BAAI/bge-reranker-base")

bm25_retriever
→ BM25 检索器对象

vector_retriever
→ 向量检索器对象

bm25_docs
→ BM25 返回的文档列表

vector_docs
→ 向量检索返回的文档列表

In [ ]:
documents = DirectoryLoader(
    knowledge_path,
    glob="**/*.md",
    loader_cls=UnstructuredMarkdownLoader,
).load()
chunks = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100
).split_documents(documents)

vectorstore = Chroma.from_documents(chunks, embedding_model)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 8}) # 创建向量检索器 / Top-K 配置方式不同
bm25_retriever = BM25Retriever.from_documents(chunks) # 创建 bm25检索器
bm25_retriever.k = 8 # 表示创建向量检索器，并且每次返回最相似的 8 个文档块。

BM25 结果：8 个文档块
向量检索结果：8 个文档块
合并后：最多 16 个，去重后可能更少
Reranker：对合并结果重新排序
最终：取前 5 个

```python
bm25_docs = [
    Document(
        page_content="文本转换可以改变文本的语言、语气或格式。",
        metadata={"source": "transforming.md"},
    ),
    Document(
        page_content="文本转换常用于翻译和语气调整。",
        metadata={"source": "transforming.md"},
    ),
]
```

In [ ]:
question = "总结文本转换这篇文章的主要观点、方法和示例"
bm25_docs = bm25_retriever.invoke(question)
vector_docs = vector_retriever.invoke(question)

# 合并并去重：同一个文档块只保留一份
candidates = {}
for doc in bm25_docs + vector_docs:
    key = (doc.metadata.get("source"), doc.page_content) # 文档来源 + 文档内容 生成唯一标识
    candidates[key] = doc
candidate_docs = list(candidates.values())
print(f"BM25={len(bm25_docs)}, vector={len(vector_docs)}, candidates={len(candidate_docs)}")

In [ ]:
# Cross-Encoder 同时读取问题和文档，给候选文档打相关性分数
pairs = [(question, doc.page_content) for doc in candidate_docs]
scores = reranker.predict(pairs)
ranked_docs = [
    doc for _, doc in sorted(
        zip(scores, candidate_docs),
        key=lambda item: item[0], # 对，item 是自定义的临时变量名，可以随便取
        reverse=True,
    )
]
final_docs = ranked_docs[:5]

for score, doc in sorted(zip(scores, candidate_docs), reverse=True, key=lambda item: item[0])[:5]:
    print(round(float(score), 4), doc.metadata.get("source"))

这段代码的作用是：

> 按照 Reranker 分数从高到低，重新排列文档。

```python
ranked_docs = [
    doc
    for _, doc in sorted(
        zip(scores, candidate_docs),
        key=lambda item: item[0],
        reverse=True,
    )
]
```

假设：

```python
scores = [0.3, 0.9, 0.6]

candidate_docs = [
    "文档A",
    "文档B",
    "文档C",
]
```

### 第一步：`zip`

```python
zip(scores, candidate_docs)
```

将分数和文档一一配对：

```python
[
    (0.3, "文档A"),
    (0.9, "文档B"),
    (0.6, "文档C"),
]
```

### 第二步：`sorted`

```python
sorted(
    ...,
    key=lambda item: item[0],
    reverse=True,
)
```

按照每个 tuple 的第一个元素，也就是分数排序：

```python
[
    (0.9, "文档B"),
    (0.6, "文档C"),
    (0.3, "文档A"),
]
```

其中：

```python
item[0]
```

表示分数；

```python
item[1]
```

表示文档。

```python
reverse=True
```

表示降序排列，分数高的在前面。

### 第三步：列表推导式

```python
[
    doc
    for _, doc in sorted(...)
]
```

只取排序后的文档，不要分数：

```python
[
    "文档B",
    "文档C",
    "文档A",
]
```

这里的 `_` 表示：

```python
这个位置的分数不用
```

## 等价的普通写法

```python
scored_docs = zip(scores, candidate_docs)

sorted_docs = sorted(
    scored_docs,
    key=lambda item: item[0],
    reverse=True,
)

ranked_docs = []

for score, doc in sorted_docs:
    ranked_docs.append(doc)
```

最终：

```python
ranked_docs
```

就是按相关性从高到低排列的文档列表。然后可以取前几个：

```python
final_docs = ranked_docs[:5]
```

In [ ]:
context = "\n\n".join(doc.page_content for doc in final_docs)
prompt = ChatPromptTemplate.from_template(
    "只根据上下文回答问题，覆盖多个相关片段，不要只总结一个示例。"
    "\n上下文：{context}\n问题：{question}"
)
answer = (prompt | llm | StrOutputParser()).invoke({"context": context, "question": question})
print(answer)